# SERIES DE TIEMPO

Entorno: Databricks

In [0]:
%pip install prophet xgboost pmdarima==2.0.4

In [0]:
import mlflow
import pandas as pd
import numpy as np

from pmdarima import auto_arima

import matplotlib.pyplot as plt
import seaborn as sns


from prophet import Prophet
from xgboost import XGBRegressor

from sklearn.model_selection import TimeSeriesSplit
from statsmodels.datasets import get_rdataset
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.ensemble import StackingRegressor, RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression

In [0]:
import warnings
warnings.filterwarnings("ignore")


In [0]:
air_passengers = get_rdataset("AirPassengers").data
air_passengers 

In [0]:
def decimal_year_to_month(decimal_year):
    # El año es la parte entera
    year = int(decimal_year)
    
    # La fracción del año se multiplica por 12 y se le suma 1 para obtener el mes
    month_fraction = decimal_year - year
    month = int(round(month_fraction * 12 + 1))
    
    # Devolvemos el formato YYYY-MM con el mes formateado a dos dígitos
    return f"{year}-{month:02d}"

air_passengers['time'] = air_passengers['time'].apply(decimal_year_to_month)


In [0]:
air_passengers

In [0]:
sns.lineplot(x=air_passengers.index, y='value', data=air_passengers)
plt.title('Air Passengers')
plt.xlabel('Time')

In [0]:
plot_acf(air_passengers['value'], lags= len(air_passengers['value'])-1) 
plt.show()

## Decomposicion de Series de Tiempo

In [0]:
decomposition = seasonal_decompose(air_passengers['value'], model='multiplicative', period=12)
decomposition.plot()
plt.show()

## MLflow

In [0]:
mlflow.set_registry_uri("databricks")
mlflow.create_experiment("/Users/ronaldo91929@gmail.com/Series de Tiempo - 28 - 10 - 2025")

In [0]:
mlflow.set_experiment("/Users/ronaldo91929@gmail.com/Series de Tiempo - 28 - 10 - 2025")

In [0]:
train_air_passengers = air_passengers[:int(len(air_passengers)*0.8)]
test_air_passengers = air_passengers[int(len(air_passengers)*0.8):]

## AutoARIMA

In [0]:
with mlflow.start_run(run_name="AUTOARIMA v2") as run:
    model = auto_arima(train_air_passengers['value'], seasonal=True, m=12)
    predictions = model.predict(n_periods=len(test_air_passengers))
    mae =  mean_absolute_error(
        test_air_passengers['value'].values, predictions
    )
    mse = mean_squared_error(
        test_air_passengers['value'].values, predictions
    )
    rmse = mse ** 0.5
    mape = mean_absolute_percentage_error(
        test_air_passengers['value'].values, predictions
    )

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)

    sns.lineplot(x=train_air_passengers.index, y='value', data=train_air_passengers, label='Train')
    sns.lineplot(x=test_air_passengers.index, y='value', data=test_air_passengers, label='Test')
    sns.lineplot(x=test_air_passengers.index, y=predictions, label='value')
    mlflow.log_figure(plt.gcf(), "predictions.png")

## Exponential Smoothing - HotWinters

In [0]:

with mlflow.start_run(run_name="HOTLWINTERS ETS") as run:
    model = ExponentialSmoothing(train_air_passengers['value'], trend='add', seasonal='mul', seasonal_periods=12).fit()
    predictions = model.forecast(len(test_air_passengers))
    mae =  mean_absolute_error(
        test_air_passengers['value'].values, predictions
    )
    mse = mean_squared_error(
        test_air_passengers['value'].values, predictions
    )
    rmse = mse ** 0.5
    mape = mean_absolute_percentage_error(
        test_air_passengers['value'].values, predictions
    )

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)

    sns.lineplot(x=train_air_passengers.index, y='value', data=train_air_passengers, label='Train')
    sns.lineplot(x=test_air_passengers.index, y='value', data=test_air_passengers, label='Test')
    sns.lineplot(x=test_air_passengers.index, y=predictions, label='Predictions')
    mlflow.log_figure(plt.gcf(), "predictions.png")

## PROPHET

In [0]:
train_air_passengers.rename(columns={'Month': 'ds', '#Passengers': 'y'})

In [0]:
# 2. Renombrar las columnas al formato de Prophet
TRAIN_PROPHET_DF = train_air_passengers.rename(columns={
    'time': 'ds',   # 'time' se convierte en 'ds' (datestamp)
    'value': 'y'    # 'value' se convierte en 'y' (el valor a predecir)
})

TEST_PROPHET_DF = test_air_passengers.rename(columns={
    'time': 'ds', 
    'value': 'y'    
})

TRAIN_PROPHET_DF['ds'] = pd.to_datetime(TRAIN_PROPHET_DF['ds'])
TEST_PROPHET_DF['ds'] = pd.to_datetime(TEST_PROPHET_DF['ds'])

TRAIN_PROPHET_DF.head()

In [0]:
TRAIN_PROPHET_DF = train_air_passengers.rename(columns={'Month': 'ds', '#Passengers': 'y'})

In [0]:
with mlflow.start_run(run_name="PROPHET") as run:
    model = Prophet(seasonality_mode='multiplicative') #adivite - 3 hyperparametros son auto en prophet (daily_seasonality, weekly_seasonality, yearly_seasonality)
    # Ajustar changepoint_prior_scale (tendencia) y seasonality_prior_scale (estacionalidad) para mejorar el modelo
    model.fit(TRAIN_PROPHET_DF)
    
    predictions = model.predict(TEST_PROPHET_DF)
    y_predictions = predictions['yhat'].values

    mae =  mean_absolute_error(
        test_air_passengers['value'].values, y_predictions
    )
    mse = mean_squared_error(
        test_air_passengers['value'].values, y_predictions
    )
    rmse = mse ** 0.5
    mape = mean_absolute_percentage_error(
        test_air_passengers['value'].values, y_predictions
    )

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)

    sns.lineplot(x=train_air_passengers.index, y='value', data=train_air_passengers, label='Train')
    sns.lineplot(x=test_air_passengers.index, y='value', data=test_air_passengers, label='Test')
    sns.lineplot(x=test_air_passengers.index, y=y_predictions, label='Predictions')
    mlflow.log_figure(plt.gcf(), "predictions.png")

## XGBoost

In [0]:
xgboost_df = air_passengers.rename(columns={'time': 'ds', 'value': 'y'})
xgboost_df["LAG1"] = xgboost_df["y"].shift(1)
xgboost_df["LAG2"] = xgboost_df["y"].shift(2)
xgboost_df["LAG3"] = xgboost_df["y"].shift(3)
xgboost_df["LAG4"] = xgboost_df["y"].shift(4)

# --- 2. Ingeniería de Características de Tiempo (¡El paso clave!) ---
# Convertir a datetime para poder extraer información
xgboost_df["ds"] = pd.to_datetime(xgboost_df["ds"])

# Extraer el mes y el año como nuevas características numéricas

xgboost_df["mes"] = xgboost_df["ds"].dt.month
xgboost_df["ano"] = xgboost_df["ds"].dt.year

xgboost_df = xgboost_df.dropna()

X, y = xgboost_df.drop(["y", "ds"], axis=1), xgboost_df["y"]

#Dividir datos de esta forma para series de tiempo
split_point = int(len(X) * 0.8)

X_train, X_test = X[:split_point], X[split_point:]
y_train, y_test = y[:split_point], y[split_point:]

X_train.head()

In [0]:
with mlflow.start_run(run_name="XGBoost basic") as run:
    xgb = XGBRegressor().fit(X_train, y_train) 
    y_pred = xgb.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    mape = mean_absolute_percentage_error(y_test, y_pred)

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)

    sns.lineplot(x=X_train.index, y=y_train,  label='Train')
    sns.lineplot(x=X_test.index, y=y_test, label='Test')
    sns.lineplot(x=X_test.index, y=y_pred, label='Predictions')
    mlflow.log_figure(plt.gcf(), "predictions.png")

In [0]:
xgboost_df = air_passengers.rename(columns={'time': 'ds', 'value': 'y'})
xgboost_df["LAG1"] = xgboost_df["y"].shift(1)
xgboost_df["LAG2"] = xgboost_df["y"].shift(2)
xgboost_df["LAG3"] = xgboost_df["y"].shift(3)
xgboost_df["LAG4"] = xgboost_df["y"].shift(4)

# --- 2. Ingeniería de Características de Tiempo (¡El paso clave!) ---
# Convertir a datetime para poder extraer información
xgboost_df["ds"] = pd.to_datetime(xgboost_df["ds"])

# Extraer el mes y el año como nuevas características numéricas

xgboost_df["mes"] = xgboost_df["ds"].dt.month
xgboost_df["ano"] = xgboost_df["ds"].dt.year
#xgboost_df["semana_del_ano"] = xgboost_df["ds"].dt.isocalendar().week.astype(int)
#xgboost_df["trimestre"] = xgboost_df["ds"].dt.quarter

# --- 3. ¡NUEVO! Crear Window Features (Ventanas Móviles) ---
# Media móvil de los últimos 2 y 3 meses

xgboost_df['media_movil_2M'] = xgboost_df['y'].shift(1).rolling(window=2).mean()
xgboost_df['media_movil_3M'] = xgboost_df['y'].shift(1).rolling(window=3).mean()
xgboost_df['media_movil_6M'] = xgboost_df['y'].shift(1).rolling(window=6).mean()

# Desviación estándar móvil de los últimos 3 meses (volatilidad)
xgboost_df['std_movil_2M'] = xgboost_df['y'].shift(1).rolling(window=2).std()
xgboost_df['std_movil_3M'] = xgboost_df['y'].shift(1).rolling(window=3).std()
xgboost_df['std_movil_6M'] = xgboost_df['y'].shift(1).rolling(window=6).std()

# Nota: Usamos .shift(1) antes de .rolling() para evitar la fuga de datos.
# Así, el cálculo para hoy solo usa información hasta ayer.

xgboost_df = xgboost_df.dropna()

X, y = xgboost_df.drop(["y", "ds"], axis=1), xgboost_df["y"]

#Dividir datos de esta forma para series de tiempo
split_point = int(len(X) * 0.8)

X_train, X_test = X[:split_point], X[split_point:]
y_train, y_test = y[:split_point], y[split_point:]

X_train.head()

In [0]:
with mlflow.start_run(run_name="XGBoost Complete") as run:
    xgb = XGBRegressor().fit(X_train, y_train) 
    y_pred = xgb.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    mape = mean_absolute_percentage_error(y_test, y_pred)

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)

    sns.lineplot(x=X_train.index, y=y_train,  label='Train')
    sns.lineplot(x=X_test.index, y=y_test, label='Test')
    sns.lineplot(x=X_test.index, y=y_pred, label='Predictions')
    mlflow.log_figure(plt.gcf(), "predictions.png")

## Emsamble

In [0]:
class FeatureEngineering:
    """
    Una clase para crear características de series de tiempo.
    """
    def __init__(self, df):
        self.df = df.copy()

    def __add_time_features(self):
        """Añade características basadas en el tiempo (año, mes)."""
        self.df["time"] = pd.to_datetime(self.df["time"])
        self.df["ano"] = self.df["time"].dt.year
        self.df["mes"] = self.df["time"].dt.month
    
    def __add_window_features(self, lista: list):
        """Añade características de ventana móvil (media y std)."""
        for window in lista:
            self.df[f'media_movil_{window}M'] = self.df['value'].shift(1).rolling(window=window).mean()
            self.df[f'std_movil_{window}M'] = self.df['value'].shift(1).rolling(window=window).std()
    
    def __add_lag_features(self, num_lags=4):
        """Añade características de lag (valores pasados)."""
        for i in range(1, num_lags+1):
            self.df[f'LAG{i}'] = self.df['value'].shift(i)
    
    def run(self, lista_window=[], nun_lags=4):
        """
        Ejecuta el pipeline completo de creación de características.
        """
        self.__add_time_features()
        self.__add_lag_features(nun_lags)
        if len(lista_window) > 0:
            self.__add_window_features(lista_window)
        
        self.df = self.df.dropna()
        self.df.rename(columns={'time': 'ds', 'value': 'y'}, inplace=True)
        return self.df
        

In [0]:
class DataSplitter:
    """
    Una clase para dividir datos de series de tiempo.
    """
    def __init__(self):
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
    
    def __split_data(self, porcentaje=0.8, df=None):
        """
        Divide los datos en conjuntos de entrenamiento y prueba.
        """
        X,y = df.drop(["y", "ds"], axis=1), air_df["y"]
        split_point = int(len(air_df) * porcentaje)

        self.X_train, self.X_test = X[:split_point], X[split_point:]
        self.y_train, self.y_test = y[:split_point], y[split_point:]

    def run(self, df=None, porcentaje=0.8):
        """
        Ejecuta el pipeline completo de división de datos.
        """
        self.__split_data(porcentaje, df)
        return self.X_train, self.X_test, self.y_train, self.y_test


In [0]:
class ProphetWrapper(BaseEstimator, RegressorMixin):
    # Heredamos de las clases base de Scikit-learn para compatibilidad
    
    def __init__(self, seasonality_mode='additive'):
        self.seasonality_mode = seasonality_mode
        self.model = None

    def fit(self, X, y):
        df_prophet = pd.DataFrame({'ds': pd.to_datetime(X["ano"].astype(str)+"-"+X["mes"].astype(str)),
                                   'y': y})
        
        # Creamos y entrenamos el modelo Prophet
        self.model = Prophet(seasonality_mode=self.seasonality_mode)
        self.model.fit(df_prophet)
        return self # .fit() siempre debe devolver self

    def predict(self, X):
        # Para predecir, Prophet necesita un DataFrame futuro con la columna 'ds'.
        future = pd.DataFrame({'ds': pd.to_datetime(X["ano"].astype(str)+"-"+X["mes"].astype(str))})
        
        # Hacemos la predicción
        forecast = self.model.predict(future)
        
        # Devolvemos solo los valores de la predicción ('yhat') como un array de NumPy
        return forecast['yhat'].values
    




class ArimaWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, seasonal=True, m=12):
        self.seasonal = seasonal
        self.m = m
        self.model = None

    def fit(self, X, y):
        self.model = auto_arima(y, seasonal=self.seasonal, m=self.m)
        return self

    def predict(self, X):
        return self.model.predict(n_periods=len(X))
    




class ExponentialSmoothingWrapper(BaseEstimator, RegressorMixin):
    """
    Clase adaptadora (wrapper) para hacer que el modelo ExponentialSmoothing
    de statsmodels sea compatible con el API de Scikit-learn.
    """
    def __init__(self, trend='add', seasonal='add', seasonal_periods=12):
        # Guardamos los hiperparámetros que queremos usar
        self.trend = trend
        self.seasonal = seasonal
        self.seasonal_periods = seasonal_periods
        self.model_ = None # Aquí guardaremos el modelo entrenado

    def fit(self, X, y):
        """
        Entrena el modelo. Scikit-learn pasa X e y.
        ExponentialSmoothing es univariado, por lo que solo usa 'y'.
        """
        # Creamos la instancia del modelo con los datos de entrenamiento 'y'
        # y los hiperparámetros guardados.
        model_instance = ExponentialSmoothing(
            pd.Series(y, index=X.index), 
            trend=self.trend, 
            seasonal=self.seasonal, 
            seasonal_periods=self.seasonal_periods,
        )
        
        # Entrenamos el modelo y lo guardamos
        self.model_ = model_instance.fit()
        
        # El método .fit() siempre debe devolver 'self'
        return self

    def predict(self, X):
        """
        Realiza una predicción. Scikit-learn pasa 'X', que representa
        los puntos futuros que queremos predecir.
        """
        # Verificamos si el modelo ha sido entrenado
        if self.model_ is None:
            raise RuntimeError("Debes entrenar el modelo primero con .fit()")
            
        # El método .forecast() necesita el número de pasos a predecir,
        # que es simplemente la longitud del DataFrame X de entrada.
        n_periods = len(X)
        
        # Hacemos el pronóstico
        predictions = self.model_.forecast(steps=n_periods)
        
        # Devolvemos las predicciones como un array de NumPy
        return predictions.values

In [0]:
class EnsambleTimeSeriesProcessor:
    def __init__(self, base_models: list, meta_model):
        self.base_models = base_models
        self.meta_model = meta_model
        self.final_base_models = None

    def __meta_features(self, X_train, y_train, n_splits = 2):
        tscv = TimeSeriesSplit(n_splits=2)
        meta_features = np.zeros((X_train.shape[0], len(self.base_models)))
        first_prediction_index = None

        for i, (name, model) in enumerate(self.base_models):
            print(f"Generando predicciones out-of-fold para: {name}")
    
            # Recorre cada pliegue de la validación cruzada
            for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):

                if fold == 0 and i == 0:
                    first_prediction_index = val_idx[0]

                # Divide los datos en entrenamiento y validación para ESTA vuelta
                X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
                y_tr        = y_train.iloc[train_idx]

                # Entrena el modelo base SÓLO con los datos de entrenamiento de este pliegue
                model.fit(X_tr, y_tr)
                
                # Haz predicciones SÓLO en el pliegue de validación
                preds = model.predict(X_val)
                
                # Guarda las predicciones en las posiciones correctas del array
                meta_features[val_idx, i] = preds


        meta_features = meta_features[first_prediction_index:] #Para limpiar los primeros ceros
        y_features = y_train[first_prediction_index:]
        X_features= pd.DataFrame(meta_features, columns=[name for name, _ in base_models])

        print("\nMeta-features (dataset para el meta-modelo) creadas con éxito.")
        return X_features, y_features
    
    def __MetaModelFit(self,X_train, y_train,  n_splits = 2 ):
        """ --- 3. Entrenar el Meta-Modelo ---
           Ahora usamos las predicciones generadas como características de entrenamiento."""
        X_features, y_features = self.__meta_features(X_train, y_train, n_splits)
        self.meta_model.fit(X_features, y_features)
        print("Meta-modelo entrenado con éxito sin datalake.")

    def fit(self, X_train, y_train, n_splits = 2):
        #Entrenamos el MetaModel
        self.__MetaModelFit(X_train, y_train, n_splits= 2)

        """ 4. Evaluación Final ---
        # Para evaluar, primero re-entrenamos los modelos base con Todo el X_train
        # para que tengan la máxima información posible. """

        print("\nRe-entrenando modelos base con todos los datos de entrenamiento...")
        self.final_base_models = []
        for name, model in self.base_models:
            model.fit(X_train, y_train)
            self.final_base_models.append(model)

    def predict(self, X_test):
        """ --- 5. Predicciones ---
        # Ahora podemos hacer predicciones usando los modelos base entrenados con Todo el X_train.
        """
        meta_features_test = np.zeros((X_test.shape[0], len(self.final_base_models)))

        for i, model in enumerate(self.final_base_models):
            meta_features_test[:, i] = model.predict(X_test)

        X_test_ensamble_df = pd.DataFrame(meta_features_test, columns=[name for name, _ in base_models])

        # Finalmente, usamos el meta-modelo para la predicción final
        y_pred = self.meta_model.predict(X_test_ensamble_df)

        return y_pred
        

In [0]:
def evaluate(y_test, y_pred, run_name, ensamble_processor):
    with mlflow.start_run(run_name= run_name) as run:
        print("Definiendo el ensamble y registrando parámetros...")
        mlflow.log_param("meta_model", ensamble_processor.meta_model.__class__.__name__)
        mlflow.log_param("base_models", [name for name, _ in ensamble_processor.base_models])
        mlflow.log_param("n_splits_cv", 2)

        for name, model in ensamble_processor.base_models:
            params = {f"{name}__{k}": v for k, v in model.get_params().items()}
            mlflow.log_params(params)

        print("\nRegistrando el modelo de ensamble como un artefacto...")
        mlflow.sklearn.log_model(
            sk_model=ensamble_processor,
            artifact_path="ensemble_time_series_model"
        )

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = mse ** 0.5
        mape = mean_absolute_percentage_error(y_test, y_pred)

        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("MSE", mse)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("MAPE", mape)

        print("MAE", mae)
        print("MSE", mse)
        print("RMSE", rmse)
        print("MAPE", mape)

        sns.lineplot(x=X_train.index, y=y_train,  label='Train')
        sns.lineplot(x=X_test.index, y=y_test, label='Test')
        sns.lineplot(x=X_test.index, y=y_pred, label='Predictions')
        mlflow.log_figure(plt.gcf(), "predictions.png")

In [0]:
fe = FeatureEngineering(air_passengers)
air_df = fe.run(lista_window=[2,4], nun_lags=4)
air_df.head()

In [0]:
X_train, X_test, y_train, y_test = DataSplitter().run(df=air_df, porcentaje = 0.8)
X_train.head()

In [0]:
base_models = [
    ('arima', ArimaWrapper(seasonal=True)),
    ('prophet', ProphetWrapper(seasonality_mode='multiplicative')),
    ('ets', ExponentialSmoothingWrapper(seasonal='mul', trend='add')),
    ('xgboost', XGBRegressor(random_state=42))
]

meta_model = LinearRegression()

model_emsamble = EnsambleTimeSeriesProcessor(base_models=base_models, meta_model=meta_model)
model_emsamble.fit(X_train, y_train)
y_pred = model_emsamble.predict(X_test)

In [0]:
evaluate(y_test, y_pred, run_name= "Stacking_v1", ensamble_processor=model_emsamble)

### Tambien se puede usar Stacking, pero hay DataLake, NO ES RECOMENDABLE PARA TIME SERIES

In [0]:
estimators = [
    ('arima', ArimaWrapper(seasonal=True, m=12)),
    ('prophet', ProphetWrapper(seasonality_mode='multiplicative')),
    ('ets', ExponentialSmoothingWrapper(seasonal='mul', trend='add',seasonal_periods=12)),
    ('xgboost', XGBRegressor()),
]


final_estimator = LinearRegression()

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=final_estimator,
)


print("Entrenando el modelo de Stacking (esto puede tardar)...")
# El .fit() orquesta todo: la validación cruzada, el entrenamiento de los base
# y el entrenamiento del meta-modelo.
stacking_model.fit(X_train, y_train)
print("¡Entrenamiento completado!")

# --- 5. Evaluar el Ensamble Final ---
# Hacemos predicciones en el conjunto de prueba que nunca ha sido visto.
y_pred = stacking_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mape = mean_absolute_percentage_error(y_test, y_pred)

print("MAE", mae)
print("MSE", mse)
print("RMSE", rmse)
print("MAPE", mape)